# TEI XML Parser for TML (Latin) Music Theory Treatises

### Overview
This notebook processes TEI (Text Encoding Initiative) XML files of Latin music theory treatises and creates a vector database for semantic search using LangChain and ChromaDB.

### Key Features of This Notebook
This notebook now uses an intelligent incremental update approach:

1. CONFIGURATION TRACKING (db_config.json)
   - Tracks embedding model, chunk size, and other settings
   - Only recreates DB when breaking changes are detected
   - Breaking changes: embedding model or chunk size changes

2. FILE CHANGE DETECTION (file_hashes.json)
   - Tracks MD5 hash of each source XML file
   - Only reprocesses files that have changed
   - Skips unchanged files automatically

3. DOCUMENT ID MANAGEMENT
   - Each chunk gets a unique, deterministic ID
   - Allows updating existing documents without duplicates
   - Format: MD5(source_file_page_number_chunk_index)

4. INCREMENTAL UPDATES
   - When a file changes, old documents are deleted first
   - New documents are added with same IDs if content unchanged
   - Prevents duplicate embeddings

### Common Operations:

```python
# Add or update files
process_xml_files()  # Only processes changed files

# Force complete rebuild (rare)
process_xml_files(force_reprocess=True)

# View database statistics
get_db_stats()

# Remove a specific file
delete_source_file("MARLU9.xml")

# Search the database
results = vector_store.similarity_search("your query here", k=5)

# Search with scores
results = vector_store.similarity_search_with_score("your query", k=5)
```


### Main Steps:
1. **Import Libraries** - Load necessary Python packages
2. **Configure Database** - Set up ChromaDB with intelligent update tracking
3. **Parse TEI XML** - Extract metadata and text from treatises
4. **Create Embeddings** - Generate vector embeddings using OpenAI
5. **Store & Query** - Save to ChromaDB and explore the database

---

### Step 1: Import Required Libraries and API Key

In [1]:
# Standard library imports
import glob
import hashlib
import json
import os
import re
import shutil
import getpass
from pathlib import Path

# Third-party imports
import pandas as pd
from bs4 import BeautifulSoup, NavigableString

# LangChain imports
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

/Users/rfreedma/anaconda3/envs/lang/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---

### Step 2: Configure OpenAI API Key and ChromaDB Settings

We use OpenAI's embedding model to convert text into vectors for semantic search.

In [2]:
# Prompt for OpenAI API key with password masking
print("Please enter your OpenAI API key:")
openai_api_key = getpass.getpass("API Key: ")

# Set as environment variable
os.environ["OPENAI_API_KEY"] = openai_api_key

# Verify it was set (show only first/last few characters for security)
if openai_api_key:
    masked_key = f"{openai_api_key[:7]}...{openai_api_key[-4:]}"
    print(f"✓ API key set successfully: {masked_key}")
else:
    print("✗ No API key entered")



Please enter your OpenAI API key:
✓ API key set successfully: sk-proj...NogA


In [3]:


# Configuration for database schema and settings--these will be passed to all the relevant components below
DB_CONFIG = {
    "version": "1.0",
    "embedding_model": "text-embedding-3-small",
    "chunk_size": 2000,
    "chunk_overlap": 300,
    "collection_name": "tml_latin"
}

db_path = Path('./chroma-db_latin')
config_path = db_path / 'db_config.json'

# Check if we need to recreate the database
should_recreate = False

if db_path.exists() and config_path.exists():
    # Load existing config
    with open(config_path, 'r') as f:
        existing_config = json.load(f)
    
    # Check for breaking changes
    if (existing_config.get('embedding_model') != DB_CONFIG['embedding_model'] or
        existing_config.get('chunk_size') != DB_CONFIG['chunk_size']):
        print(f"⚠️  Breaking changes detected:")
        print(f"   Old: {existing_config}")
        print(f"   New: {DB_CONFIG}")
        should_recreate = True
    else:
        print(f"✓ Using existing database - configuration unchanged")
        print(f"  Will perform incremental updates only")
elif not db_path.exists():
    print(f"✓ Creating new database at {db_path}")
    should_recreate = True
else:
    print(f"⚠️  Database exists but no config found - will recreate")
    should_recreate = True

# Delete database only if necessary
if should_recreate and db_path.exists():
    shutil.rmtree(db_path)
    print(f"✓ Deleted existing database at {db_path}")

# Create directory if needed
db_path.mkdir(exist_ok=True)

# Save current configuration
with open(config_path, 'w') as f:
    json.dump(DB_CONFIG, f, indent=2)

# Initialize embeddings
embeddings = OpenAIEmbeddings(model=DB_CONFIG['embedding_model'])

# Initialize Chroma vector store
vector_store = Chroma(
    collection_name=DB_CONFIG['collection_name'],
    embedding_function=embeddings,
    persist_directory=str(db_path)
)

# Configure text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=DB_CONFIG['chunk_size'],
    chunk_overlap=DB_CONFIG['chunk_overlap'],
    length_function=len,
    is_separator_regex=False
)

✓ Using existing database - configuration unchanged
  Will perform incremental updates only


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


---

### Step 3: Initialize ChromaDB Vector Store

#### What is a Vector Database?
A vector database stores text as numerical vectors (embeddings) that capture semantic meaning. This allows us to find relevant passages based on meaning, not just keyword matching.

#### Intelligent Update Strategy
This notebook implements a **smart incremental update system**:
- **Configuration Tracking**: Detects breaking changes (embedding model, chunk size)
- **File Change Detection**: Only reprocesses modified XML files
- **No Unnecessary Rebuilds**: Saves time and API costs

The system creates two tracking files:
- `db_config.json` - Stores database configuration
- `file_hashes.json` - Tracks which files have been processed

In [4]:
# metadata extraction function for the TML Latin music theory files
# the metadata will be used in the chroma db and also in the csv file for the Streamlit app as a record of our sources


def parse_century_to_years(date_str):
    """
    Convert century strings to year ranges.
    
    Examples:
        "14th" -> (1300, 1399)
        "15th" -> (1400, 1499)
        "6th-8th" -> (500, 799)
        "13th" -> (1200, 1299)
    """
    if not date_str or date_str == "Unknown Date":
        return (1200, 1599)  # Default broad range for medieval/Renaissance
    
    # Handle range like "6th-8th"
    range_match = re.match(r'(\d+)(?:st|nd|rd|th)-(\d+)(?:st|nd|rd|th)', date_str)
    if range_match:
        start_century = int(range_match.group(1))
        end_century = int(range_match.group(2))
        return ((start_century - 1) * 100, end_century * 100 - 1)
    
    # Handle single century like "14th"
    single_match = re.match(r'(\d+)(?:st|nd|rd|th)', date_str)
    if single_match:
        century = int(single_match.group(1))
        return ((century - 1) * 100, century * 100 - 1)
    
    # Try to find a year in the string
    year_match = re.findall(r'\b(1[0-6]\d{2})\b', date_str)
    if year_match:
        year = int(year_match[0])
        return (year, year)
    
    # Default fallback
    return (1200, 1599)


def extract_metadata(soup, html_path):
    # Convert to Path object if it's a string
    if isinstance(html_path, str):
        html_path = Path(html_path)
    
    # The TitleInfo is in the nav element, not in main
    metadata_div = soup.find("titleStmt")
    if not metadata_div:
        return None

    else:

        # Add error handling for each span element
        title = metadata_div.find("title")
        author = metadata_div.find("author")
        date_elem = metadata_div.find("date")
        file_id = metadata_div.find("title", {"type": "abbrev"})

        title = title.get_text(strip=True) if title else "Unknown Title"
        author = author.get_text(strip=True) if author else "Unknown Author"
        
        # Remove parentheses and strip whitespace from date
        if date_elem:
            date_raw = date_elem.get_text(strip=True).replace("(", "").replace(")", "").strip()
            # Add "century" to century patterns like "13th" or "6th-8th"
            if re.match(r'^\d+(?:st|nd|rd|th)(?:-\d+(?:st|nd|rd|th))?$', date_raw):
                date = f"{date_raw} century"
            else:
                date = date_raw
            # Convert century to year range using raw value (without "century" suffix)
            date_start, date_end = parse_century_to_years(date_raw)
        else:
            date = "Unknown Date"
            date_start, date_end = parse_century_to_years(date)

        return {
            "filename": html_path.name,
            "title": title,
            "author": author,
            "date": date,
            "date_start": date_start,
            "date_end": date_end,
            "citation": f"chtml{html_path.name}"
        }

# # main page content extraction
# main page content extraction
# def extract_pages_and_text(xml_content, xml_path):
#     """
#     Extract page numbers and their associated text from TEI XML.
    
#     Page numbers are in <span class="tei pb"><span class="tei pbSpan">page X</span></span>
#     Page content follows each page break as sibling elements until the next page break.
    
#     Args:
#         xml_content: Either a string containing TEI XML or a BeautifulSoup object
#         xml_path: Path to the XML file

#     Returns:
#         List of dictionaries with 'pageNumber', 'pageText', and metadata
#     """
#     # Check if xml_content is already a BeautifulSoup object
#     if isinstance(xml_content, BeautifulSoup):
#         soup = xml_content
#     else:
#         soup = BeautifulSoup(xml_content, 'xml')

#     metadata = extract_metadata(soup, xml_path)
    
#     # Find all page break tags 
#     page_breaks = soup.find_all('pb')
    
#     results = []
    
#     for i, pb in enumerate(page_breaks):
#         # Extract page number from the pb tag
#         page_number = re.findall(r'\d+', pb.get_text(strip=True))[0] if re.findall(r'\d+', pb.get_text(strip=True)) else f"{i+1}"

#         # Collect ALL text content that follows this page break until the next one
#         # Strategy: Get all text nodes between the two page breaks in document order
        
#         # Get the next page break to know where to stop
#         next_pb = page_breaks[i + 1] if i + 1 < len(page_breaks) else None
        
#         # Collect all NavigableStrings (text nodes) between page breaks
#         page_text_parts = []
        
#         # Start after current page break
#         for element in pb.next_elements:
#             # Stop if we hit the next page break
#             if next_pb and element == next_pb:
#                 break
            
#             # Only collect NavigableString objects (actual text nodes)
#             # This avoids double-counting when we have nested tags
#             # note that we are NOT yet collecting information about images, and figures, just text
            
#             if isinstance(element, NavigableString):
#                 text = str(element).strip()
#                 if text:
#                     page_text_parts.append(text)
        
#         page_text = ' '.join(page_text_parts)
#         # this is the assembled text content for this page number
#         result_entry = {
#             'pageNumber': page_number,
#             'pageText': page_text
#         }
        
#         # Attach metadata if available
#         if metadata:
            
#             result_entry.update(metadata)
        
#         results.append(result_entry)
    
#     return results

def extract_pages_and_text(xml_content, xml_path, chunk_size, chunk_overlap):
    """
    Extract text from TEI XML and create chunks with page span metadata.
    
    Instead of creating one chunk per page, this creates chunks of roughly uniform size
    and tracks which pages each chunk spans (e.g., "ii-iii" or "29-31").
    
    Args:
        xml_content: Either a string containing TEI XML or a BeautifulSoup object
        xml_path: Path to the XML file
        chunk_size: Target size for chunks in characters (from DB_CONFIG)
        chunk_overlap: Number of characters to overlap between chunks (from DB_CONFIG)
        
    Returns:
        List of dictionaries with 'pageSpan', 'chunkText', and metadata
    """
    # Check if xml_content is already a BeautifulSoup object
    if isinstance(xml_content, BeautifulSoup):
        soup = xml_content
    else:
        soup = BeautifulSoup(xml_content, 'xml')

    metadata = extract_metadata(soup, xml_path)
    
    # Find all page break tags 
    page_breaks = soup.find_all('pb')
    
    # Step 1: Extract all pages with their text
    pages = []
    
    for i, pb in enumerate(page_breaks):
        # Extract page number from the pb tag
        n_attr = pb.get('n', '')
        page_number = re.findall(r'\d+', n_attr)[0] if re.findall(r'\d+', n_attr) else f"{i+1}"


        # page_number = re.findall(r'\d+', pb.get_text(strip=True))[0] if re.findall(r'\d+', pb.get_text(strip=True)) else f"{i+1}"
        
        # Get the next page break to know where to stop
        next_pb = page_breaks[i + 1] if i + 1 < len(page_breaks) else None
        
        # Collect all NavigableStrings (text nodes) between page breaks
        page_text_parts = []
        
        # Start after current page break
        for element in pb.next_elements:
            # Stop if we hit the next page break
            if next_pb and element == next_pb:
                break
            
            # Only collect NavigableString objects (actual text nodes)
            if isinstance(element, NavigableString):
                text = str(element).strip()
                if text:
                    page_text_parts.append(text)
        
        page_text = ' '.join(page_text_parts)
        
        # Store the page number and its text
        if page_text:  # Only include pages with actual text
            pages.append({
                'page': page_number,
                'text': page_text
            })
    
    # Step 2: Build full text and track page boundaries
    full_text_parts = []
    page_boundaries = []  # Track where each page starts and ends in full_text
    current_pos = 0
    
    for page_info in pages:
        text = page_info['text']
        start_pos = current_pos
        end_pos = current_pos + len(text)
        
        page_boundaries.append({
            'page': page_info['page'],
            'start': start_pos,
            'end': end_pos
        })
        
        full_text_parts.append(text)
        current_pos = end_pos + 1  # +1 for the space we'll add when joining
    
    full_text = ' '.join(full_text_parts)
    
    # Step 3: Create chunks and determine page spans
    results = []
    start = 0
    
    while start < len(full_text):
        # Define chunk boundaries
        end = min(start + chunk_size, len(full_text))
        chunk_text = full_text[start:end]
        
        # Find pages that overlap with this chunk
        pages_in_chunk = []
        for page_bound in page_boundaries:
            # Check if this page overlaps with the chunk [start, end)
            if not (page_bound['end'] <= start or page_bound['start'] >= end):
                pages_in_chunk.append(page_bound['page'])
        
        # Format page span
        if len(pages_in_chunk) == 0:
            page_span = "unknown"
        elif len(pages_in_chunk) == 1:
            page_span = pages_in_chunk[0]
        else:
            page_span = f"{pages_in_chunk[0]}-{pages_in_chunk[-1]}"
        
        # Create result entry
        result_entry = {
            'pageSpan': page_span,
            'chunkText': chunk_text
        }
        
        # Attach metadata if available
        if metadata:
            result_entry.update(metadata)
        
        results.append(result_entry)
        
        # Move to next chunk with overlap
        start = end - chunk_overlap if end < len(full_text) else end
    
    return results

---

### Step 4: Define TEI XML Parsing Functions

These functions extract structured data from TEI XML files:

`extract_metadata(soup, html_path)`
Extracts bibliographic information from the TEI header:
- Title of the treatise
- Author name
- Date/century
- Source filename

`extract_pages_and_text(xml_content, xml_path)`
Extracts page-by-page text content:
- Identifies page breaks (`<pb>` tags)
- Collects all text between page breaks
- Preserves document structure
- Returns list of pages with metadata

---

### Step 5: Test Text Extraction (Optional)

**Purpose**: This cell is for testing/debugging only. It extracts text and metadata from XML files without creating embeddings.


**Note**: You can skip this cell and go directly to `process_xml_files()` for normal operation.

In [5]:
# run this on all files in the source directory:  this is just to get the text, not the vector db
tei_dir = Path("latin_sources")
for xml_path in tei_dir.glob("*.xml"):
    with xml_path.open("r", encoding="utf-8") as handle:
        xml_content = handle.read()    

        # results = extract_pages_and_text(html_content, html_path)
        results = extract_pages_and_text(xml_content, 
                                         xml_path, 
                                         chunk_size=DB_CONFIG["chunk_size"], 
                                         chunk_overlap=DB_CONFIG["chunk_overlap"])

        

In [6]:
# just to check one result

results[0]

{'pageSpan': '1-2',
 'chunkText': 'Petro de Cruce Ambianensi Capitulum I Incipit Tractatus de Tonis a Magistro Petro de Cruce [fol. 52v] Dicturi de tonis, primo videndum est quid sit tonus et unde dicatur. Tonus ut hic accipitur est quaedam regula quae de omni cantu in fine diiudicat. Octo sunt toni. Primus vocatur autentus protus, id est auctoritate primus. Secundus vocatur plaga proti, id est pars primi. Tertius vocatur autentus deuterus, id est auctoritate secundus. Quartus vocatur plaga deuteri, id est pars eius. Quintus vocatur autentus tritus, id est auctoritate tertius. Sextus vocatur plaga triti, id est pars eius. Septimus vocatur autentus tetrardus, id est auctoritate quartus. Octavus vocatur plaga tetrardi, id est pars eius. Quatuor sunt litterae finales, scilicet D.E.F.G., et dicuntur finales quia isti toni regulariter finiuntur in ipsis. Tres sunt litterae affinales, scilicet a.b.c., et dicuntur affinales quia suppleant vices aliarum quatuor; et hoc est quando aliqui tonoru

---
## Get the Metadata DataFrame


In [7]:
# test metaddata df
tei_dir = Path("latin_sources")
metadata = []
for html_path in tei_dir.glob("*.xml"):
    with html_path.open("r", encoding="utf-8") as handle:
        html_content = handle.read()    
        one_document_metadata = extract_metadata(BeautifulSoup(html_content, 'xml'), html_path)  
        metadata.append(one_document_metadata)   
metadata_df = pd.DataFrame(metadata)
metadata_df.to_csv("latin_tei_metadata.csv", index=False)
metadata_df

,filename,title,author,date,date_start,date_end,citation
0,ORNMUS2.xml,Musice active micrologus liber secundus,"Ornithoparchus, Andreas",16th century,1500,1599,chtmlORNMUS2.xml
1,MONEPI.xml,Epitoma utriusque musicae practicae,"Monetarius, Stefan",16th century,1500,1599,chtmlMONEPI.xml
2,WALREGU_MLBLL763.xml,Regule Magistri. Thome Walsingham. De figuris ...,"Walsingham, Thomas",14th century,1300,1399,chtmlWALREGU_MLBLL763.xml
3,RUDQUA.xml,Quaestiones in musica,Rudolf of St. Trond,12th century,1100,1199,chtmlRUDQUA.xml
4,ANOAM.xml,Ars musice,Anonymous,15th century,1400,1499,chtmlANOAM.xml
...,...,...,...,...,...,...,...
934,ENGDEM1.xml,"De musica, tractatus primus",Engelbertus Admontensis,14th century,1300,1399,chtmlENGDEM1.xml
935,REGDHI.xml,De harmonica institutione,Regino Prumiensis,9th-11th century,800,1099,chtmlREGDHI.xml
936,MARLUC14.xml,"Lucidarium, tractatus quartus decimus",Marchetus de Padua,14th century,1300,1399,chtmlMARLUC14.xml
937,ANOTRUT.xml,Tractatus de musica et tonarius,Anonymous,12th century,1100,1199,chtmlANOTRUT.xml


## Functions to Process all the Files and Create/Update the ChromaDB

In [8]:

def generate_document_id(source_file, page_number, chunk_index):
    """Generate a unique, deterministic ID for each document chunk."""
    id_string = f"{source_file}_{page_number}_chunk_{chunk_index}"
    return hashlib.md5(id_string.encode()).hexdigest()

def get_file_hash(filepath):
    """Get MD5 hash of a file to detect changes."""
    hash_md5 = hashlib.md5()
    with open(filepath, "rb") as f:
        for chunk in iter(lambda: f.read(4096), b""):
            hash_md5.update(chunk)
    return hash_md5.hexdigest()

def load_file_hashes():
    """Load previously processed file hashes."""
    hash_file = db_path / 'file_hashes.json'
    if hash_file.exists():
        with open(hash_file, 'r') as f:
            return json.load(f)
    return {}

def save_file_hashes(hashes):
    """Save file hashes to track what's been processed."""
    hash_file = db_path / 'file_hashes.json'
    with open(hash_file, 'w') as f:
        json.dump(hashes, f, indent=2)

def process_xml_files(xml_dir='latin_sources', 
                      force_reprocess=False,
                      chunk_size=DB_CONFIG["chunk_size"], 
                      chunk_overlap=DB_CONFIG["chunk_overlap"]):
    """
    Process all TEI XML files in the specified directory.
    
    Args:
        xml_dir: Directory containing XML files
        force_reprocess: If True, reprocess all files regardless of changes
    """
    xml_files = glob.glob(os.path.join(xml_dir, '*.xml'))
    
    if not xml_files:
        print(f"No TEI XML files found in {xml_dir}")
        return
    
    # Load existing file hashes to detect changes
    existing_hashes = load_file_hashes()
    new_hashes = {}
    
    total_chunks = 0
    total_pages = 0
    files_processed = 0
    files_skipped = 0
    files_updated = 0
    
    for filepath in xml_files:
        try:
            filename = os.path.basename(filepath)
            current_hash = get_file_hash(filepath)
            new_hashes[filename] = current_hash
            
            # Skip if file hasn't changed (unless force_reprocess is True)
            if not force_reprocess and filename in existing_hashes:
                if existing_hashes[filename] == current_hash:
                    print(f"⊙ {filename} - No changes, skipping")
                    files_skipped += 1
                    continue
                else:
                    print(f"↻ {filename} - File changed, updating...")
                    files_updated += 1
                    # Delete old documents for this file
                    source_id = f"chtml{filename}"
                    try:
                        vector_store.delete(where={"source": source_id})
                        print(f"  Deleted old documents for {filename}")
                    except Exception as e:
                        print(f"  Note: Could not delete old documents: {e}")
            else:
                print(f"+ {filename} - New file, processing...")
            
            with open(filepath, 'r', encoding='utf-8') as f:
                xml_content = f.read()
            
            # parse XML to return the pages in the source document, with metadata
            # note that the metadata extraction is done inside extract_pages_and_text
            pages = extract_pages_and_text(xml_content, 
                                           Path(filepath),
                                           chunk_size=chunk_size,
                                         chunk_overlap=chunk_overlap)
            
            if not pages:
                print(f"  Warning: No pages found in {filepath}")
                continue
            
            file_chunks = 0
            chunk_counter = 0
            all_chunk_ids = []
            all_chunks = []
            
            # chunk each page in the source document separately
            for page in pages:
                # Create metadata for this page, combining document and page metadata
                page_metadata = {
                    "title": page['title'],
                    "author": page['author'],
                    "date": page['date'],
                    "date_start": page['date_start'],
                    "date_end": page['date_end'],
                    "citation": page['citation'],
                    "page_range": page['pageSpan'],
                    "filename": filename
                }
                
                # Split page text into chunks
                chunks = text_splitter.create_documents(
                    texts=[page['chunkText']],
                    metadatas=[page_metadata]
                )
                
                # Generate IDs for each chunk
                for chunk in chunks:
                    chunk_id = generate_document_id(
                        page['citation'], 
                        page['pageSpan'], 
                        chunk_counter
                    )
                    all_chunk_ids.append(chunk_id)
                    all_chunks.append(chunk)
                    chunk_counter += 1
                
                file_chunks += len(chunks)
            
            # Add all chunks for this file at once with IDs
            if all_chunks:
                vector_store.add_documents(
                    documents=all_chunks,
                    ids=all_chunk_ids
                )
            
            total_chunks += file_chunks
            total_pages += len(pages)
            files_processed += 1
            
            print(f'  ✓ Title: {page["title"][:80]}...' if len(page["title"]) > 80 else f'  ✓ Title: {page["title"]}')
            print(f'    Author: {page["author"]} | Date: {page["date_start"]}-{page["date_end"]}')
            print(f'    Pages: {len(pages)} | Chunks: {file_chunks}')
            
        except Exception as e:
            print(f"✗ Error processing {filepath}: {str(e)}")
            import traceback
            traceback.print_exc()
            continue
    
    # Save the new hashes
    save_file_hashes(new_hashes)
    
    print(f'\n{"="*50}')
    print(f'ChromaDB Processing Complete')
    print(f'{"="*50}')
    print(f'Total files: {len(xml_files)}')
    print(f'  New/Updated: {files_processed}')
    print(f'  Skipped (unchanged): {files_skipped}')
    print(f'  Changed: {files_updated}')
    print(f'Total pages processed: {total_pages}')
    print(f'Total chunks added: {total_chunks}')

---

## Step 6: Process XML Files and Create Vector Database

### What Happens Here:

1. **File Hash Tracking**: Computes MD5 hash of each XML file to detect changes
2. **Smart Processing**: 
   - ⊙ **Skips** unchanged files
   - ↻ **Updates** modified files (deletes old, adds new)
   - \+ **Processes** new files
3. **Text Chunking**: Splits long pages into 2000-character chunks with 300-char overlap
4. **ID Generation**: Creates deterministic IDs for each chunk (prevents duplicates)
5. **Embedding Creation**: Sends chunks to OpenAI for vector embedding
6. **Database Storage**: Stores embeddings and metadata in ChromaDB

### Understanding Chunks vs Pages:
- **Page**: A logical division from the original document (marked by `<pb>` tags)
- **Chunk**: A piece of text ≤2000 characters for optimal embedding
- One page may create multiple chunks if text is long

In [9]:
# Process XML files - only processes new or changed files by default
# Use force_reprocess=True to reprocess everything
process_xml_files(xml_dir='latin_sources', 
                  force_reprocess=True, 
                  chunk_size=DB_CONFIG["chunk_size"],
                  chunk_overlap=DB_CONFIG["chunk_overlap"])

+ ORNMUS2.xml - New file, processing...
  ✓ Title: Musice active micrologus liber secundus
    Author: Ornithoparchus, Andreas | Date: 1500-1599
    Pages: 30 | Chunks: 30
+ MONEPI.xml - New file, processing...
  ✓ Title: Epitoma utriusque musicae practicae
    Author: Monetarius, Stefan | Date: 1500-1599
    Pages: 40 | Chunks: 40
+ WALREGU_MLBLL763.xml - New file, processing...
  ✓ Title: Regule Magistri. Thome Walsingham. De figuris compositis. et non compositis. et ...
    Author: Walsingham, Thomas | Date: 1300-1399
    Pages: 14 | Chunks: 14
+ RUDQUA.xml - New file, processing...
  ✓ Title: Quaestiones in musica
    Author: Rudolf of St. Trond | Date: 1100-1199
    Pages: 67 | Chunks: 67
+ ANOAM.xml - New file, processing...
  ✓ Title: Ars musice
    Author: Anonymous | Date: 1400-1499
    Pages: 2 | Chunks: 2
+ GUIMICB_MBBR2784.xml - New file, processing...
  ✓ Title: Micrologus
    Author: Guido d'Arezzo | Date: 800-1099
    Pages: 29 | Chunks: 29
+ TINCPT3.xml - New file, proc

In [10]:
# for checking page and character counts


pages = results
        # Calculate lengths of all PageText entries
page_lengths = [len(page['chunkText']) for page in pages]
avg_length = sum(page_lengths) / len(page_lengths)
    
# print(f"\n{filename}:")
print(f"  Total pages: {len(pages)}")
print(f"  Average pageText length: {avg_length:.2f} characters")
print(f"  Min length: {min(page_lengths)} characters")
print(f"  Max length: {max(page_lengths)} characters")


  Total pages: 5
  Average pageText length: 1926.60 characters
  Min length: 1633 characters
  Max length: 2000 characters


In [11]:
# another check for page refs and length

for page in pages:
    if len(page['chunkText']) < 1000:
        print(page['author'])
        print(page['chunkText'])

        print(f"Page {page['pageSpan']} is very short: {len(page['chunkText'])} characters")

In [12]:
# Helper functions for database management

def get_db_stats():
    """Get statistics about the current database."""
    all_docs = vector_store.get()
    
    if not all_docs or 'metadatas' not in all_docs:
        print("Database is empty")
        return
    
    total_docs = len(all_docs['ids'])
    
    # Collect statistics
    authors = set()
    citations = set()
    dates = set()
    
    for metadata in all_docs['metadatas']:
        if metadata:
            if 'author' in metadata:
                authors.add(metadata['author'])
            if 'citation' in metadata:
                citations.add(metadata['citation'])
            if 'date' in metadata:
                dates.add(metadata['date'])
    
    print(f"Database Statistics:")
    print(f"  Total chunks: {total_docs}")
    print(f"  Unique authors: {len(authors)}")
    print(f"  Unique citations: {len(citations)}")
    print(f"  Date range: {sorted(dates)}")
    print(f"\nAuthors: {sorted(authors)}")
    
    return {
        'total_docs': total_docs,
        'authors': sorted(authors),
        'citations': sorted(citations),
        'dates': sorted(dates)
    }

def delete_source_file(source_filename):
    """Delete all documents from a specific source file."""
    source_id = f"chtml{source_filename}"
    try:
        vector_store.delete(where={"source": source_id})
        print(f"✓ Deleted all documents from {source_filename}")
        
        # Also remove from file hashes
        hashes = load_file_hashes()
        if source_filename in hashes:
            del hashes[source_filename]
            save_file_hashes(hashes)
            print(f"✓ Removed {source_filename} from tracking")
    except Exception as e:
        print(f"✗ Error deleting documents: {e}")

# Get current database statistics
get_db_stats()

Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


Database Statistics:
  Total chunks: 24443
  Unique authors: 225
  Unique citations: 939
  Date range: ['12th century', '13th century', '14th century', '15th century', '16th century', '17th century', '3rd-5th century', '6th-8th century', '9th-11th century']

Authors: ['Aaron, Petrus', 'Abbot Guido', 'Adamus de Fulda', 'Adelboldus', 'Aegidius Zamorensis, Iohannes', 'Aegidius de Murino', 'Agricola, Martinus', 'Al-Farabi', 'Alcuinus, Flaccus', 'Amerus', 'Ancina Fossaniensis, Juvenal', 'Anonymous', 'Anonymous 1', 'Anonymous 2', 'Anonymous 3', 'Anonymous 4', 'Anonymous 5', 'Anonymous 6', 'Anonymous 7', 'Anonymous I', 'Anonymous II', 'Anonymous III', 'Anonymous IV', 'Anonymous OP', 'Anonymous V', 'Anonymous VI', 'Anonymous VII', 'Anonymous VIII', 'Anonymous X', 'Anonymous XI', 'Anonymous XII', 'Anselmus, Georgius', 'Antonius de Luca', 'Aquinas [Ps.], Thomas', 'Aribo', 'Aristotle', 'Arnulphus de Sancto Gilleno', 'Augustinus, Aurelius', 'Aurelianus Reomensis', 'Beda [Ps.]', 'Bede [Ps.]', 'Bern

{'total_docs': 24443,
 'authors': ['Aaron, Petrus',
  'Abbot Guido',
  'Adamus de Fulda',
  'Adelboldus',
  'Aegidius Zamorensis, Iohannes',
  'Aegidius de Murino',
  'Agricola, Martinus',
  'Al-Farabi',
  'Alcuinus, Flaccus',
  'Amerus',
  'Ancina Fossaniensis, Juvenal',
  'Anonymous',
  'Anonymous 1',
  'Anonymous 2',
  'Anonymous 3',
  'Anonymous 4',
  'Anonymous 5',
  'Anonymous 6',
  'Anonymous 7',
  'Anonymous I',
  'Anonymous II',
  'Anonymous III',
  'Anonymous IV',
  'Anonymous OP',
  'Anonymous V',
  'Anonymous VI',
  'Anonymous VII',
  'Anonymous VIII',
  'Anonymous X',
  'Anonymous XI',
  'Anonymous XII',
  'Anselmus, Georgius',
  'Antonius de Luca',
  'Aquinas [Ps.], Thomas',
  'Aribo',
  'Aristotle',
  'Arnulphus de Sancto Gilleno',
  'Augustinus, Aurelius',
  'Aurelianus Reomensis',
  'Beda [Ps.]',
  'Bede [Ps.]',
  'Bernardus abbatis Clareuallis',
  'Bernelinus',
  'Berno Augiensis',
  'Bertrandus Prudentius',
  'Beurhusius, Fredericus',
  'Bianchini, Francesco',
  'Boe

---

## Step 7: Explore Database Contents

These cells demonstrate what's stored in the database and how to access it.

### 7.1: View Sample Metadata

Each chunk in the database has metadata that describes its source.

In [15]:
# Get a sample of documents from the database
sample_docs = vector_store.get(limit=10)

# Display metadata from first 3 chunks
print("=" * 60)
print("SAMPLE METADATA FROM DATABASE")
print("=" * 60)

for i, metadata in enumerate(sample_docs['metadatas'][0:10], 1):
    print(f"\n📄 Chunk {i}:")
    print(f"   Title: {metadata.get('title', 'N/A')}")
    print(f"   Author: {metadata.get('author', 'N/A')}")
    print(f"   Date: {metadata.get('date', 'N/A')}")
    print(f"   Pages: {metadata.get('page_range', 'N/A')}")
    print(f"   Source File: {metadata.get('filename', 'N/A')}")
    print(f"   Source ID: {metadata.get('citation', 'N/A')}")

SAMPLE METADATA FROM DATABASE

📄 Chunk 1:
   Title: Lucidarium, tractatus decimussextus
   Author: Marchetus de Padua
   Date: 14th
   Pages: 121
   Source File: MARLU16.xml
   Source ID: chtmlMARLU16.xml

📄 Chunk 2:
   Title: [Musica]
   Author: Anonymous 4
   Date: 13th
   Pages: 22-23
   Source File: ANO4MUS.xml
   Source ID: chtmlANO4MUS.xml

📄 Chunk 3:
   Title: [Musica]
   Author: Anonymous 4
   Date: 13th
   Pages: 23-24
   Source File: ANO4MUS.xml
   Source ID: chtmlANO4MUS.xml

📄 Chunk 4:
   Title: [Musica]
   Author: Anonymous 4
   Date: 13th
   Pages: 24-25
   Source File: ANO4MUS.xml
   Source ID: chtmlANO4MUS.xml

📄 Chunk 5:
   Title: [Musica]
   Author: Anonymous 4
   Date: 13th
   Pages: 24-25
   Source File: ANO4MUS.xml
   Source ID: chtmlANO4MUS.xml

📄 Chunk 6:
   Title: [Musica]
   Author: Anonymous 4
   Date: 13th
   Pages: 25-26
   Source File: ANO4MUS.xml
   Source ID: chtmlANO4MUS.xml

📄 Chunk 7:
   Title: [Musica]
   Author: Anonymous 4
   Date: 13th
   Pages: 26

### 7.2: View Sample Text Chunks

See what the actual text chunks look like.

In [23]:
# Display text content from sample chunks
print("=" * 60)
print("SAMPLE TEXT CHUNKS")
print("=" * 60)

for i, (doc_text, metadata) in enumerate(zip(sample_docs['documents'][:3], sample_docs['metadatas'][:3]), 1):
    print(f"\n📝 Chunk {i}:")
    print(f"   Source: {metadata.get('title', 'N/A')} (Page {metadata.get('page_number', 'N/A')})")
    print(f"   Length: {len(doc_text)} characters")
    print(f"   Text Preview:")
    print(f"   {'-' * 55}")
    # Show first 300 characters
    preview = doc_text[:300] + "..." if len(doc_text) > 300 else doc_text
    print(f"   {preview}")
    print()

SAMPLE TEXT CHUNKS

📝 Chunk 1:
   Source: Lucidarium, tractatus decimussextus (Page 1)
   Length: 1112 characters
   Text Preview:
   -------------------------------------------------------
   Incipit tractatus XVI. De Musico et Cantore. Musicus dicitur ille, testante Boetio, cui adest facultas secundum speculationem et rationem ipsius scientiae musicae, de modis atque rhythmis, deque generibus cantilenarum (iudicare). Omnis enim ars seu disciplina honorabiliorem naturaliter habet ratione...


📝 Chunk 2:
   Source: [Musica] (Page 1)
   Length: 1677 characters
   Text Preview:
   -------------------------------------------------------
   Capitulum primum Pars prima (A 59r, C 56v) Cognita modulatione melorum secundum viam octo troporum et secundum usum et consuetudinem fidei catholicae nunc habendum est de mensuris eorundem secundum longitudinem et brevitatem, prout antiqui tractaverunt, ut magister Leo et alii plurimi plenius iuxta ...


📝 Chunk 3:
   Source: [Musica] (Page 2)
   Length

### 7.3: View Sample Vectors (Embeddings)

**What are vectors?** Numerical representations of text meaning. OpenAI's `text-embedding-3-large` creates 3072-dimensional vectors.

**Why vectors?** They enable semantic search - finding text with similar *meaning*, not just matching keywords.

In [48]:
# Display information about the embedding vectors
import numpy as np

if 'embeddings' in sample_docs and sample_docs['embeddings']:
    print("=" * 60)
    print("SAMPLE EMBEDDING VECTORS")
    print("=" * 60)
    
    for i, (embedding, metadata) in enumerate(zip(sample_docs['embeddings'][:3], sample_docs['metadatas'][:3]), 1):
        print(f"\n🔢 Chunk {i} Vector:")
        print(f"   Source: {metadata.get('title', 'N/A')} (Page {metadata.get('page_number', 'N/A')})")
        print(f"   Vector Dimensions: {len(embedding)}")
        print(f"   Vector Type: {type(embedding)}")
        print(f"   First 10 values: {embedding[:10]}")
        print(f"   Vector stats:")
        print(f"      - Min value: {min(embedding):.6f}")
        print(f"      - Max value: {max(embedding):.6f}")
        print(f"      - Mean value: {np.mean(embedding):.6f}")
        print(f"      - Std deviation: {np.std(embedding):.6f}")
else:
    print("Note: Embeddings not included in sample. Use include=['embeddings'] when calling get().")
    print("\nTo retrieve with embeddings:")
    print("sample_with_embeddings = vector_store.get(limit=3, include=['embeddings', 'documents', 'metadatas'])")

Note: Embeddings not included in sample. Use include=['embeddings'] when calling get().

To retrieve with embeddings:
sample_with_embeddings = vector_store.get(limit=3, include=['embeddings', 'documents', 'metadatas'])


## One Sample Document from the Vector Store

In [57]:
sample_with_embeddings = vector_store.get(limit=1, include=['embeddings', 'documents', 'metadatas'])

# how many dimensions in the embedding vector?
len(sample_with_embeddings['embeddings'][0])

3072

### 7.4: Perform a Semantic Search

Demonstrate how to search the database by meaning.

In [50]:
# Example: Search for passages about musical intervals
query = "diapente and diatessaron intervals"

print("=" * 60)
print(f"SEMANTIC SEARCH EXAMPLE")
print("=" * 60)
print(f"\nQuery: '{query}'")
print("\nTop 3 most relevant passages:\n")

# Perform similarity search
results = vector_store.similarity_search(query, k=3)

for i, doc in enumerate(results, 1):
    print(f"{'='*60}")
    print(f"Result {i}:")
    print(f"   Title: {doc.metadata.get('title', 'N/A')}")
    print(f"   Author: {doc.metadata.get('author', 'N/A')}")
    print(f"   Page: {doc.metadata.get('page_number', 'N/A')}")
    print(f"   Date: {doc.metadata.get('date', 'N/A')}")
    print(f"\n   Text excerpt:")
    print(f"   {'-'*55}")
    # Show first 400 characters
    preview = doc.page_content[:400] + "..." if len(doc.page_content) > 400 else doc.page_content
    print(f"   {preview}")
    print()

SEMANTIC SEARCH EXAMPLE

Query: 'diapente and diatessaron intervals'

Top 3 most relevant passages:

Result 1:
   Title: Musices opusculum, tractatus primus
   Author: Burtius, Nicolaus
   Page: page 46
   Date: 15th

   Text excerpt:
   -------------------------------------------------------
   De dyatesseron. Diatesseron etenim quattuor est vocum: ac duorum tonorum: vniusque semitonij minoris acceruatio Dicta namque a dia quod est de vel per: et tessera quattuor. eo quod sit de quatuor sonis effecta. Habet enim tres speties. Prima enim cadit inter.A. graue et .d. Secunda vero inter .[sqb]. graue et .E. Tertia autem inter .C. graue et F. Uel prima cadit inter .D. graue et G. Secunda inte...

Result 2:
   Title: De institutione musica, liber secundus
   Author: Boethius, Anicius Manlius Severinus
   Page: page 36
   Date: 6th-8th

   Text excerpt:
   -------------------------------------------------------
   diapente sibimet iunctae efficiunt triplum, diatessaron vero et tonus diapente

### 7.5: Search with Similarity Scores

See the actual similarity scores to understand how close the matches are.

In [51]:
# Search with similarity scores (lower distance = more similar)
query = "musical consonance and dissonance"

print("=" * 60)
print(f"SEMANTIC SEARCH WITH SCORES")
print("=" * 60)
print(f"\nQuery: '{query}'")
print("\nResults ranked by similarity:\n")

# Perform similarity search with scores
results_with_scores = vector_store.similarity_search_with_score(query, k=5)

for i, (doc, score) in enumerate(results_with_scores, 1):
    print(f"{'='*60}")
    print(f"Result {i} - Similarity Score: {score:.4f}")
    print(f"   Title: {doc.metadata.get('title', 'N/A')}")
    print(f"   Author: {doc.metadata.get('author', 'N/A')}")
    print(f"   Page: {doc.metadata.get('page_number', 'N/A')}")
    print(f"\n   Text excerpt (first 250 chars):")
    print(f"   {'-'*55}")
    preview = doc.page_content[:250] + "..." if len(doc.page_content) > 250 else doc.page_content
    print(f"   {preview}")
    print()

print("\n💡 Note: Lower scores indicate higher similarity (distance metric)")
print("   Typical range: 0.0 (identical) to 2.0 (very different)")

SEMANTIC SEARCH WITH SCORES

Query: 'musical consonance and dissonance'

Results ranked by similarity:

Result 1 - Similarity Score: 0.9002
   Title: Musices opusculum, tractatus primus
   Author: Burtius, Nicolaus
   Page: page 22

   Text excerpt (first 250 chars):
   -------------------------------------------------------
   et probrie consonantia est diapente: diapasson cum diapente: et bisdiapason. vt infra constabit cum de cantu commixto pertractabimus. Dicitur antem consonantia a consequendo vt Hysidoro placet. quia consequendo organizat voces suas. Euphonia etiam id...

Result 2 - Similarity Score: 0.9379
   Title: Lucidarium, tractatus secundus
   Author: Marchetus de Padua
   Page: page 7

   Text excerpt (first 250 chars):
   -------------------------------------------------------
   Caput X. De proportionibus consonantiarum et dissonantiarum. Quoniam musica est superius definita, quod est scientia, quae in numeris et proportionibus consistit, viso et declarato, quomodo in e

---

## Summary and Next Steps

### What You've Built:
✅ A vector database of Latin music theory treatises  
✅ Semantic search capability (search by meaning, not keywords)  
✅ Intelligent incremental update system (saves time and costs)  
✅ Complete metadata tracking (author, title, date, page numbers)  

### Key Concepts:
- **TEI XML**: Text Encoding Initiative format for digital scholarly texts
- **Embeddings**: Numerical representations of text meaning (3072-dimensional vectors)
- **Chunks**: Text pieces ≤2000 characters for optimal embedding
- **Vector Database**: Stores embeddings for fast similarity search
- **Semantic Search**: Finding relevant passages by meaning, not just keywords


vector_store

In [ ]:
# one document in the vector store

vector_store.get(limit=1, include=['metadatas', 'documents'])

{'ids': ['aa5d506b0f35708c05339eb6fc3005af'],
 'embeddings': None,
 'documents': ['Incipit tractatus IV. Caput I. De proportionibus. Primo, quid proportio. Proportio est quaedam habitudo duorum terminorum ad invicem, vel distantia duorum terminorum inter se. Sed notandum est, quod aliud est proportio, et aliud est proportionalitas: nam proportio, ut praedicitur, est de duobus terminis; proportionalitas autem non minus quam de tribus sumi potest: dicitur enim proportio pars numeralis in musica. Caput II. De proportionibus, quot sint. Proportiones in musica, in quibus consonantiae consistunt, sunt sex, scilicet sesquitertia, sesquialtera, dupla, dupla superbipartiens, tripla et quadrupla. Proportiones vero, in quibus membra consonantiarum consistunt, sunt tres, scilicet sesquioctava, sesquisexta decima, et sesquidecimaseptima: de quibus omnibus est videndum. Caput III. De sesquitertia proportione. Sesquitertia proportio est, quando maior numerus comparatus minori continent ipsum totum, e